# 05d — Attention Head-Count Sensitivity Ablation (`h in {1, 2, 4, 8, 12}`)

**Reviewer Question Addressed:** *"How was the number of heads set to 4 decided?"*

We fix the backbone to **Domain MLM (`xlm-roberta-base`, 3 epochs)** and the task heads, class weights, learning rates, and splits (`runA`, `runB`) identically to `MLM+attn`, and sweep all valid multi-head divisors of $d = 768$:
- `h = 1` ($d_k = 768$), `h = 2` ($d_k = 384$), `h = 4` ($d_k = 192$, proposed default), `h = 8` ($d_k = 96$), `h = 12` ($d_k = 64$).

- Reuses cached `mlm_encoders/mlm_{run}_seed{seed}` checkpoints when present (zero redundant MLM retraining).
- Set `SEEDS_TO_EXEC = [42]` for a quick single-seed sweep on limited GPU quota, or keep `SEEDS_TO_EXEC = SEEDS` (`[42, 43, 44]`) for full 3-seed means/stds.

## 1. Environment & Single Config Cell

In [1]:
# !pip install -q transformers accelerate
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if not os.path.exists("eval_utils.py") and not os.path.exists("/kaggle/working/Sinhala-joint-sentiment-and-intent"):
    subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/kaggle/working/Sinhala-joint-sentiment-and-intent"], check=True)
if os.path.exists("/kaggle/working/Sinhala-joint-sentiment-and-intent"):
    os.chdir("/kaggle/working/Sinhala-joint-sentiment-and-intent")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import gc
import math
import time
import json
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    NS,
    NI,
    SEEDS,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# -- Single Config Cell -------------------------------------------------------
MODEL_NAME       = "xlm-roberta-base"
MAX_LEN          = 128
MLM_PROB         = 0.15
MLM_EPOCHS       = 3
MLM_BATCH        = 8
MLM_GRAD_ACC     = 2
MLM_LR           = 2e-5

FT_EPOCHS        = 10
FT_BATCH         = 8
FT_GRAD_ACC      = 2
FT_LR            = 2e-5
ENC_LR_MULT      = 0.1
WEIGHT_DECAY     = 0.01
DROPOUT          = 0.3
PATIENCE         = 3

HEAD_GRID        = [1, 2, 4, 8, 12]
RUNS_TO_EXEC     = ["runA", "runB"]
SEEDS_TO_EXEC    = SEEDS          # Set to [42] for quick 1-seed sweep if GPU budget is tight
NOTEBOOK_NAME    = "05d_attention_head_count_ablation"

WORK_DIR         = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV       = None
GOLD_CSV         = None
RESULTS_DIR      = os.path.join(WORK_DIR, "results")
PREDS_DIR        = os.path.join(WORK_DIR, "preds")
ENC_DIR          = os.path.join(WORK_DIR, "mlm_encoders")
for _d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE, "| HEAD_GRID =", HEAD_GRID)

Cloning into '/kaggle/working/Sinhala-joint-sentiment-and-intent'...


Environment versions: {
  "python": "3.13.16",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "scikit_learn": "1.6.1",
  "torch": "2.11.0+cu130",
  "cuda_available": "True",
  "cuda_device": "Tesla T4",
  "transformers": "5.20.2"
}
Using device: cuda | HEAD_GRID = [1, 2, 4, 8, 12]


## 2. Joint Dataset & Parameterized DualAttnPooler(heads=h) Architecture

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

class JointDS(Dataset):
    def __init__(self, df: pd.DataFrame):
        enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids = enc["input_ids"]
        self.am  = enc["attention_mask"]
        self.s   = df["sent_id"].tolist()
        self.i   = df["intent_id"].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {"input_ids": self.ids[idx], "attention_mask": self.am[idx], "sent": self.s[idx], "intent": self.i[idx]}

def collate_joint(batch):
    out = tok.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    out["sent"]   = torch.tensor([b["sent"] for b in batch], dtype=torch.long)
    out["intent"] = torch.tensor([b["intent"] for b in batch], dtype=torch.long)
    return out

class DualAttnPooler(nn.Module):
    def __init__(self, hidden: int, heads: int, dropout: float):
        super().__init__()
        assert hidden % heads == 0, f"hidden={hidden} must be divisible by heads={heads}"
        self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
        self.query_sent = nn.Parameter(torch.randn(1, 1, hidden))
        self.query_int  = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm_s = nn.LayerNorm(hidden)
        self.norm_i = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        B, kpm = H.size(0), (mask == 0)
        os_, _ = self.attn(self.query_sent.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        oi_, _ = self.attn(self.query_int.expand(B, -1, -1),  H, H, key_padding_mask=kpm)
        return self.norm_s(os_.squeeze(1)), self.norm_i(oi_.squeeze(1))

class JointHeadModel(nn.Module):
    def __init__(self, enc_path: str, heads: int):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool      = DualAttnPooler(h, heads, DROPOUT)
        self.sent_fc   = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_head = nn.Linear(h // 2, NS)
        self.int_fc    = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.int_head  = nn.Linear(h // 2, NI)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ps, pi = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(ps)), self.int_head(self.int_fc(pi))

@torch.no_grad()
def predict_joint(model, loader):
    model.eval()
    S, I = [], []
    for b in loader:
        ls, li = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        S.append(torch.softmax(ls, -1).cpu())
        I.append(torch.softmax(li, -1).cpu())
    return torch.cat(S).numpy(), torch.cat(I).numpy()

def class_weights(ids, K: int) -> torch.Tensor:
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

## 3. Head-Count Sweep Execution (HEAD_GRID = [1, 2, 4, 8, 12])

In [3]:
def get_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    for c in [os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}"), os.path.join(ENC_DIR, f"mlm_seed{seed}")]:
        if os.path.exists(os.path.join(c, "config.json")):
            print(f"Using cached domain MLM encoder: {c}")
            return c
    out = os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}")
    os.makedirs(out, exist_ok=True)
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)
    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item(); n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
        print(f"  [MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")
    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

summaries = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_texts = train_df["text"].tolist() + val_df["text"].tolist()
    assert not (set(mlm_texts) & set(test_df["text"])), "Leakage guard!"
    _enc = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    mlm_examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_enc["input_ids"], _enc["attention_mask"], _enc["special_tokens_mask"])
    ]

    train_ds, val_ds, test_ds = JointDS(train_df), JointDS(val_df), JointDS(test_df)
    val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_joint)
    test_loader = DataLoader(test_ds, batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_joint)
    W_SENT = class_weights(train_df["sent_id"].values, NS).to(DEVICE)
    W_INT  = class_weights(train_df["intent_id"].values, NI).to(DEVICE)
    val_ys, val_yi = val_df["sent_id"].values, val_df["intent_id"].values

    for heads in HEAD_GRID:
        nb_tag    = f"05d_head_ablation_h{heads}"
        model_tag = f"MLM+attn_h{heads}"
        seed_payloads = []
        for seed in SEEDS_TO_EXEC:
            enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed)
            set_all_seeds(seed)
            model = JointHeadModel(enc_path, heads=heads).to(DEVICE)
            g = torch.Generator(); g.manual_seed(seed)
            loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate_joint, generator=g)

            crit_s = nn.CrossEntropyLoss(weight=W_SENT)
            crit_i = nn.CrossEntropyLoss(weight=W_INT)
            head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
            opt = torch.optim.AdamW(
                [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
                 {"params": head_params, "lr": FT_LR}],
                weight_decay=WEIGHT_DECAY,
            )
            total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
            sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

            best_val, best_ep, best_state, bad = -1.0, 0, None, 0
            t0 = time.time()
            for ep in range(FT_EPOCHS):
                model.train(); tl = 0.0; opt.zero_grad()
                for step, b in enumerate(loader):
                    ls, li = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                    loss = crit_s(ls, b["sent"].to(DEVICE)) + crit_i(li, b["intent"].to(DEVICE))
                    (loss / FT_GRAD_ACC).backward()
                    tl += loss.item()
                    if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step(); sched.step(); opt.zero_grad()
                vs, vi = predict_joint(model, val_loader)
                fs = macro_f1_present(val_ys, vs.argmax(1), NS)
                fi = macro_f1_present(val_yi, vi.argmax(1), NI)
                fm = 0.5 * (fs + fi)
                print(f"  [h={heads} | {run} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val F1 {fs:.3f}/{fi:.3f} (mean {fm:.3f})")
                if fm > best_val:
                    best_val, best_ep, bad = fm, ep + 1, 0
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        print("  early stop")
                        break

            elapsed_s = round(time.time() - t0, 2)
            model.load_state_dict(best_state)
            ts, ti = predict_joint(model, test_loader)
            del model, opt, sched, best_state
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            payload = evaluate_and_save(
                notebook_name=nb_tag,
                model_name=model_tag,
                run=run,
                seed=seed,
                test_df=test_df,
                pred_sent_id=ts.argmax(1),
                pred_intent_id=ti.argmax(1),
                sent_probs=ts,
                intent_probs=ti,
                extra_meta={"heads": heads, "head_dim": 768 // heads, "best_epoch": best_ep, "best_val_mean_f1": round(float(best_val), 4), "train_seconds": elapsed_s},
                results_dir=RESULTS_DIR,
                preds_dir=PREDS_DIR,
            )
            seed_payloads.append(payload)

        s_row = summarize_Across_seeds(seed_payloads)
        s_row["heads"] = heads
        s_row["head_dim"] = 768 // heads
        summaries.append(s_row)

df_heads = pd.DataFrame(summaries)
display(df_heads[["run", "heads", "head_dim", "n_seeds", "sent_acc_str", "sent_f1_3cls_str", "intent_f1_str", "joint_acc_str"]])
df_heads.to_csv(os.path.join(RESULTS_DIR, f"{NOTEBOOK_NAME}_summary.csv"), index=False)
with open(os.path.join(RESULTS_DIR, f"{NOTEBOOK_NAME}_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [MLM runA seed 42] epoch 1/3 loss 4.4281
  [MLM runA seed 42] epoch 2/3 loss 3.9210
  [MLM runA seed 42] epoch 3/3 loss 3.6890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (107 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: /kaggle/working/mlm_encoders/mlm_runB_seed44
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [h=12 | runB s44] ep  1 | train 3.0593 | val F1 0.167/0.147 (mean 0.157)
  [h=12 | runB s44] ep  2 | train 2.5553 | val F1 0.434/0.480 (mean 0.457)
  [h=12 | runB s44] ep  3 | train 2.0937 | val F1 0.451/0.540 (mean 0.496)
  [h=12 | runB s44] ep  4 | train 1.8956 | val F1 0.452/0.574 (mean 0.513)
  [h=12 | runB s44] ep  5 | train 1.7892 | val F1 0.468/0.612 (mean 0.540)
  [h=12 | runB s44] ep  6 | train 1.6733 | val F1 0.485/0.618 (mean 0.551)
  [h=12 | runB s44] ep  7 | train 1.6334 | val F1 0.473/0.630 (mean 0.551)
  [h=12 | runB s44] ep  8 | train 1.5694 | val F1 0.485/0.637 (mean 0.561)
  [h=12 | runB s44] ep  9 | train 1.5136 | val F1 0.475/0.660 (mean 0.567)
  [h=12 | runB s44] ep 10 | train 1.5108 | val F1 0.476/0.659 (mean 0.567)


,run,heads,head_dim,n_seeds,sent_acc_str,sent_f1_3cls_str,intent_f1_str,joint_acc_str
0,runA,1,768,3,0.758 ± 0.025,0.695 ± 0.020,0.418 ± 0.020,0.393 ± 0.005
1,runA,2,384,3,0.756 ± 0.008,0.694 ± 0.005,0.415 ± 0.005,0.396 ± 0.011
2,runA,4,192,3,0.753 ± 0.012,0.690 ± 0.011,0.421 ± 0.019,0.398 ± 0.002
3,runA,8,96,3,0.750 ± 0.021,0.688 ± 0.015,0.421 ± 0.010,0.393 ± 0.019
4,runA,12,64,3,0.762 ± 0.011,0.699 ± 0.005,0.421 ± 0.017,0.401 ± 0.002
5,runB,1,768,3,0.754 ± 0.006,0.692 ± 0.002,0.400 ± 0.007,0.393 ± 0.005
6,runB,2,384,3,0.752 ± 0.007,0.689 ± 0.002,0.408 ± 0.001,0.394 ± 0.002
7,runB,4,192,3,0.760 ± 0.007,0.697 ± 0.004,0.414 ± 0.005,0.402 ± 0.009
8,runB,8,96,3,0.748 ± 0.011,0.683 ± 0.011,0.418 ± 0.003,0.403 ± 0.007
9,runB,12,64,3,0.739 ± 0.004,0.681 ± 0.005,0.418 ± 0.007,0.392 ± 0.004


## 4. Plot Head-Count Sensitivity Curve & Package Outputs

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
for ax, run in zip(axes, RUNS_TO_EXEC):
    sub = df_heads[df_heads["run"] == run].sort_values("heads")
    if len(sub) == 0:
        continue
    xs = sub["heads"].values
    ax.errorbar(xs, sub["sent_f1_3cls_mean"], yerr=sub["sent_f1_3cls_std"], marker="o", label="Sent F1 (3-cls)", capsize=3)
    ax.errorbar(xs, sub["intent_f1_mean"],    yerr=sub["intent_f1_std"],    marker="s", label="Intent F1 (Gold)", capsize=3)
    ax.errorbar(xs, sub["joint_acc_mean"],    yerr=sub["joint_acc_std"],    marker="^", label="Joint Acc", capsize=3)
    ax.axvline(4, color="gray", linestyle="--", linewidth=1, label="h=4 default")
    ax.set_xticks(HEAD_GRID)
    ax.set_title(f"{run}: Attention Head-Count Sweep")
    ax.set_xlabel("Number of Attention Heads (h)")
    ax.set_ylabel("Gold Benchmark Score")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "05d_head_ablation_plot.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print(f"Saved head-count sensitivity plot: {plot_path}")

zip_path = os.path.join(WORK_DIR, f"{NOTEBOOK_NAME}_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (RESULTS_DIR, PREDS_DIR):
        for fp in sorted(glob.glob(os.path.join(folder, "*05d*")) + glob.glob(os.path.join(folder, "*_h[0-9]*"))):
            zf.write(fp, arcname=os.path.relpath(fp, WORK_DIR))
print(f"Packaged outputs into: {zip_path}")

Saved head-count sensitivity plot: /kaggle/working/results/05d_head_ablation_plot.png
Packaged outputs into: /kaggle/working/05d_attention_head_count_ablation_outputs.zip
